# Bao forcing signal アブレーション・候補分類分析

pilot-v2の同一成果物に対し、forcingを候補成立に含める条件、除外する条件、補助スコアのみに使う条件を比較し、主分析候補をA/B/Cへ分類する。

## 1. 入力ZIPを展開

In [ ]:
from pathlib import Path
import zipfile

ZIP_PATH = Path('/content/pilot-v2-analysis-input.zip')
PILOT_V2 = Path('/content/pilot-v2')
PILOT_V2.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(PILOT_V2)
for name in ['observations.jsonl', 'games.json', 'manifest.json']:
    path = PILOT_V2 / name
    assert path.exists(), f'Missing: {path}'
    print(name, path.stat().st_size)


## 2. 最新リポジトリで分析実行

In [ ]:
REPO = Path('/content/bao-la-kiswahili-game')
OUTPUT = Path('/content/phase-transition-forcing-ablation')
!rm -rf {REPO} {OUTPUT}
!git clone -q https://github.com/nkkmd/bao-la-kiswahili-game.git {REPO}
!git -C {REPO} log -1 --oneline
!python {REPO}/tools/experiments/analyze-phase-transition-forcing-ablation.py \
  --input {PILOT_V2} \
  --output {OUTPUT}


## 3. 3条件とA/B/C分類を確認

In [ ]:
import json
import pandas as pd

summary = json.loads((OUTPUT / 'forcing-ablation-summary.json').read_text(encoding='utf-8'))
metrics = pd.read_csv(OUTPUT / 'forcing-ablation-summary.csv')
audit = pd.read_csv(OUTPUT / 'candidate-audit-table.csv')
display(metrics)
print(summary['overlapAtPrimaryAll100'])
print(summary['classificationAtPrimaryAll100'])
display(audit.groupby('category').head(20))


## 4. A群を優先確認

In [ ]:
a_candidates = audit[audit['category'] == 'A'].copy()
display(a_candidates.sort_values(['peakScore', 'persistence3'], ascending=False))


## 5. Google Driveへ保存

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
DESTINATION = Path('/content/drive/MyDrive/bao-la-kiswahili-game/phase-transition-analysis/pilot-v2-forcing-ablation')
DESTINATION.mkdir(parents=True, exist_ok=True)
for filename in [
    'forcing-ablation-summary.json',
    'forcing-ablation-summary.csv',
    'forcing-ablation-points.csv',
    'forcing-ablation-clusters.csv',
    'candidate-audit-table.csv',
]:
    shutil.copy2(OUTPUT / filename, DESTINATION / filename)
for path in sorted(DESTINATION.iterdir()):
    print(path.name, path.stat().st_size)


## 解釈

- A: forcingを候補成立から除外しても残り、forcing切替と同時ではない候補
- B: forcingを除外しても残るが、forcing切替と同時の候補
- C: forcingを独立特徴群として数えた場合にのみ成立する候補

重複判定は代表ply一致ではなく、同一gameId内の候補区間の重なりで行う。